In [1]:
import requests
import pandas as pd
import json, time
import os
from pathlib import Path
import boto3, json

if Path.cwd().name == "notebooks":
    os.chdir("..")

from src.config import load_config

CONFIG = load_config()

In [ ]:
region          = "eu-west-3"

iam = boto3.client("iam", region_name=region)
ec2 = boto3.client("ec2", region_name=region)
ssm = boto3.client("ssm", region_name=region)

role_name       = "airflow-ec2-role"
profil_name     = "airflow-ec2-profil"
instance_name   = "airflow-orchestrator" 
group_name      = "airflow-ec2-sg"
policy_name     = "airflow-permissions"
rds_sg          = "sg-0775b4249e3eb9757"
key_name        = "nappecast-ec2-key"

vpc_id = ec2.describe_vpcs(Filters=[{"Name": "is-default", "Values": ["true"]}])["Vpcs"][0]["VpcId"]
print(vpc_id)
my_ip = requests.get("https://checkip.amazonaws.com").text.strip()
print(my_ip)

vpc-0400adbbc2bdd55be
86.213.23.196


In [8]:
existing = ec2.describe_security_groups(
    Filters=[
        {"Name": "group-name", "Values": [group_name]},
        {"Name": "vpc-id", "Values": [vpc_id]},
    ]
)

if existing["SecurityGroups"]:
    airflow_sg_id = existing["SecurityGroups"][0]["GroupId"]
else:
    airflow_sg = ec2.create_security_group(GroupName=group_name, Description="AirFlow server", VpcId=vpc_id)
    airflow_sg_id = airflow_sg["GroupId"]

ec2.authorize_security_group_ingress(
    GroupId=airflow_sg_id,
    IpPermissions=[
        {"IpProtocol": "tcp", "FromPort": 22, "ToPort": 22, "IpRanges": [{"CidrIp": f"{my_ip}/32"}]},
        {"IpProtocol": "tcp", "FromPort": 8080, "ToPort": 8080, "IpRanges": [{"CidrIp": f"{my_ip}/32"}]},
    ],
)

print(airflow_sg_id)

sg-0d34dad32a2811316


In [ ]:
# role
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "ec2.amazonaws.com"},
        "Action": "sts:AssumeRole",
    }],
}

try:
    iam.get_role(RoleName=role_name)

except iam.exceptions.NoSuchEntityException:
    iam.create_role(
        RoleName=role_name,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Rôle EC2 pour l'orchestrateur Airflow",
    )

# policy
for policy_arn in [
    "arn:aws:iam::aws:policy/AmazonSSMManagedInstanceCore",
    "arn:aws:iam::aws:policy/CloudWatchAgentServerPolicy",
]:
    iam.attach_role_policy(RoleName=role_name, PolicyArn=policy_arn)

# permissions
inline_policy  = {
	"Version": "2012-10-17",
	"Statement": [
		{
			"Sid": "S3Sync",
			"Effect": "Allow",
			"Action": [
				"s3:GetObject",
				"s3:PutObject",
				"s3:ListBucket"
			],
			"Resource": [
				"arn:aws:s3:::nappecast",
				"arn:aws:s3:::nappecast/app/docker/airflow/*"
			]
		},
		{
			"Sid": "EMRServerlessTrigger",
			"Effect": "Allow",
			"Action": [
				"emr-serverless:StartJobRun",
				"emr-serverless:GetJobRun"
			],
			"Resource": "*"
		},
		{
			"Sid": "EC2SpotForTraining",
			"Effect": "Allow",
			"Action": [
				"ec2:RunInstances",
				"ec2:TerminateInstances",
				"ec2:DescribeInstances",
				"ec2:CreateTags"
			],
			"Resource": "*"
		}
	]
}

iam.put_role_policy(RoleName=role_name, PolicyName=policy_name, PolicyDocument=json.dumps(inline_policy))

try:
    iam.get_instance_profile(InstanceProfileName=profil_name)

except iam.exceptions.NoSuchEntityException:
    iam.create_instance_profile(InstanceProfileName=profil_name)
    iam.add_role_to_instance_profile(
        InstanceProfileName=profil_name, RoleName=role_name
    )

time.sleep(10)

In [ ]:
s3_session = boto3.Session()
s3 = s3_session.client("s3")

try:
    s3.download_file("nappecast", key, local_path)

    
except Exception as e:
    print(f"ERREUR sur la clé : {key}")
    raise

In [3]:
ami_id = ssm.get_parameter(Name="/aws/service/ami-amazon-linux-latest/al2023-ami-kernel-default-x86_64")["Parameter"]["Value"]

user_data = """#!/bin/bash
yum update -y

# --- Installation docker ---
yum install -y docker
systemctl enable docker
systemctl start docker

mkdir -p /usr/local/lib/docker/cli-plugins
curl -SL https://github.com/docker/compose/releases/latest/download/docker-compose-linux-x86_64 -o /usr/local/lib/docker/cli-plugins/docker-compose
chmod +x /usr/local/lib/docker/cli-plugins/docker-compose

BUILDX_URL=$(curl -s https://api.github.com/repos/docker/buildx/releases/latest | grep "browser_download_url.*linux-amd64\\"" | cut -d '"' -f 4 | head -n 1)
curl -SL "$BUILDX_URL" -o /usr/local/lib/docker/cli-plugins/docker-buildx
chmod +x /usr/local/lib/docker/cli-plugins/docker-buildx

yum install -y python3-pip
pip3 install boto3
"""

resp = ec2.run_instances(
    ImageId=ami_id,
    InstanceType="t3.small",
    KeyName=key_name,
    SecurityGroupIds=[airflow_sg_id],
    IamInstanceProfile={"Name": profil_name},
    UserData=user_data,
    MinCount=1, MaxCount=1,
    TagSpecifications=[{"ResourceType": "instance", "Tags": [{"Key": "Name", "Value": instance_name}]}],
)

airflow_instance_id = resp["Instances"][0]["InstanceId"]
ec2.get_waiter("instance_running").wait(InstanceIds=[airflow_instance_id])

airflow_ip = ec2.describe_instances(InstanceIds=[airflow_instance_id])["Reservations"][0]["Instances"][0]["PublicIpAddress"]
print("IP :", airflow_ip, "| instance_id :", airflow_instance_id)

alloc = ec2.allocate_address(Domain="vpc")
ec2.associate_address(
    InstanceId=airflow_instance_id,
    AllocationId=alloc["AllocationId"],
)
print("IP Elastique AirFlow :", alloc["PublicIp"])

NameError: name 'ssm' is not defined

In [ ]:
s3_session = boto3.Session()
s3 = s3_session.client("s3")

import os
from pathlib import Path

EXCLUDE_DIRS = {".git", "__pycache__", ".venv", "node_modules"}
EXCLUDE_FILES = {".env"}

project_root = Path("..").resolve() / "NappeCast"   # depuis notebook/, remonte à la racine du projet

def upload_path(s3_client, path: Path, bucket, s3_prefix, base: Path):
    if not path.exists():
        print(f"⚠️  Introuvable, ignoré : {path}")
        return
    if path.is_file():
        relative = path.relative_to(base).as_posix()
        print(f"Upload : {path} -> s3://{bucket}/{s3_prefix}/{relative}")
        s3_client.upload_file(str(path), bucket, f"{s3_prefix}/{relative}")
    elif path.is_dir():
        for root, dirs, files in os.walk(path):
            dirs[:] = [d for d in dirs if d not in EXCLUDE_DIRS]
            for file in files:
                if file in EXCLUDE_FILES:
                    continue
                local_file = Path(root) / file
                relative = local_file.relative_to(base).as_posix()
                print(f"Upload : {local_file} -> s3://{bucket}/{s3_prefix}/{relative}")
                s3_client.upload_file(str(local_file), bucket, f"{s3_prefix}/{relative}")

# uniquement ce qu'on veut déployer, pas tout le projet ni le dossier notebook/
to_upload = [
    project_root / "docker",
]

for item in to_upload:
    upload_path(s3, item, "nappecast", "app", base=project_root)

Upload : /home/ronanguilloueee/NappeCast/docker/mlflow/docker-compose.yml -> s3://nappecast/app/docker/mlflow/docker-compose.yml
Upload : /home/ronanguilloueee/NappeCast/docker/mlflow/Dockerfile -> s3://nappecast/app/docker/mlflow/Dockerfile
Upload : /home/ronanguilloueee/NappeCast/docker/airflow/docker-compose.yml -> s3://nappecast/app/docker/airflow/docker-compose.yml
Upload : /home/ronanguilloueee/NappeCast/docker/airflow/requirements.txt -> s3://nappecast/app/docker/airflow/requirements.txt
Upload : /home/ronanguilloueee/NappeCast/docker/airflow/Dockerfile -> s3://nappecast/app/docker/airflow/Dockerfile
Upload : /home/ronanguilloueee/NappeCast/docker/app_api/Caddyfile -> s3://nappecast/app/docker/app_api/Caddyfile
Upload : /home/ronanguilloueee/NappeCast/docker/app_api/docker-compose.yml -> s3://nappecast/app/docker/app_api/docker-compose.yml


#### connexion a la console : 
ssh -i "nappecast-ec2-key.pem" ec2-user@ec2-35-181-244-243.eu-west-3.compute.amazonaws.com

### Création du fichier de copie de S3 > EC2 / bash  
cat > /home/ec2-user/download_s3.py << 'EOF'  
import boto3, os  
  
s3 = boto3.client("s3", region_name="eu-west-3")  
bucket = "nappecast"  
prefix = "app/docker/airflow/"  
local_dir = "/home/ec2-user/app/docker/airflow"  
  
paginator = s3.get_paginator("list_objects_v2")  
for page in paginator.paginate(Bucket=bucket, Prefix=prefix):  
    for obj in page.get("Contents", []):  
        key = obj["Key"]  
        relative_path = key[len(prefix):]  
        if not relative_path:  
            continue  
        local_path = os.path.join(local_dir, relative_path)  
        os.makedirs(os.path.dirname(local_path), exist_ok=True)  
        s3.download_file(bucket, key, local_path)  
        print(f"Téléchargé : {key} -> {local_path}")  
EOF   
  
pip3 install boto3  

### copie des fichiers S3 > EC2
python3 /home/ec2-user/download_s3.py  

### build de l'instance docker avec passage des env.
cd /home/ec2-user/app/docker/airflow  

export_cmd=$(grep -v '^#' .env | sed "s/^/export /")
ssh -i "nappecast-ec2-key.pem" ec2-user@ec2-35-181-244-243.eu-west-3.compute.amazonaws.com "cd ~/app/docker/airflow && eval '$export_cmd' && sudo -E docker compose up -d"

## journaux
sudo docker logs airflow-s3-sync --tail 20